# Lentils × Dinomaly: inference + per-class AUROC

Load a pipeline trained by one of the `lentils_*_train_tutorial.ipynb` notebooks, run it over
the **180-frame test split**, and report:

- **overall pixel + image AUROC** (binary anomaly = any foreign object), and
- a **per-class pixel AUROC** breakdown across the 8 COCO categories, using the baked
  `class_mask` (one-vs-background per class).

> **Prerequisites**
>
> 1. Install cuvis-ai-dinomaly with the examples extra: `uv sync --extra examples`.
> 2. From the repo root, launch the notebook with `uv run jupyter lab`.
> 3. Train + save a pipeline first (run `lentils_rgb_train_tutorial.ipynb`, or the CIR /
>    adaclip-bands siblings). This notebook loads it from `PIPELINE_DIR` (set in the configuration
>    cell; defaults to the RGB run's `outputs/lentils_rgb_run/trained_models`). Edit it to evaluate
>    a different run.
>
> **Data**: `prepare_lentils_data` downloads + converts the dataset to per-frame NPZ under
> `outputs/npz_local` (reused if a train notebook already produced it there), writing a
> `universe.csv` + `splits.json`; the 180-frame test split is resolved from the splits.json.
> `TEST_LIMIT` (0 = full 180; N = first N frames) and `SMOKE_LIMIT` (frames per split when
> converting) are set in the configuration cell.

In [ ]:
# Colab bootstrap: no-op when running locally
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("In Colab Env")
    %pip install -q cuvis-ai cuvis-ai-dinomaly "cuvis-ai-dataloader[cu3s,coco]"

    import torch

    if not torch.cuda.is_available():
        print(
            "WARNING: No GPU detected. Switch via Runtime > Change runtime type > T4 GPU. "
            "Dinomaly inference on CPU is slow."
        )

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import utils
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.training import Predictor
from cuvis_ai_core.utils.node_registry import NodeRegistry
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_schemas.enums import ExecutionStage
from cuvis_ai_schemas.training import DataSplitConfig
from loguru import logger
from utils import LENTILS_CATEGORIES, prepare_lentils_data, resolve_config, resolve_pipeline

In [ ]:
# Keep the notebook output readable: log at INFO and above. Nodes emit per-step progress
# (e.g. the TensorBoard monitor) at DEBUG, a firehose across the whole test pass; raise the
# floor to INFO here.
logger.remove()
logger.add(sys.stderr, level="INFO")

device = torch.device("cpu")
# Pick the best available torch device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
logger.info(f"Using device {device}")

## Tutorial configuration

Edit these variables to choose what to evaluate and how much.

- **`PIPELINE_DIR`**: a train notebook's `trained_models` dir. Defaults to the RGB run
  (`outputs/lentils_rgb_run/trained_models`); point it at a CIR or AdaCLIP-bands run to compare
  front-ends on the identical test set.
- **`TEST_LIMIT`**: 0 evaluates the full 180-frame test split; N evaluates the first N frames.
- **`SMOKE_LIMIT`**: 0 converts all frames; N caps frames per split when preparing the NPZ (a
  fast dry-run when the data is not yet on disk).

In [ ]:
# --- Which trained run to evaluate + how much (edit these) ---------------
PIPELINE_DIR = Path("outputs/lentils_rgb_run/trained_models")  # a train notebook's output dir
TEST_LIMIT = 0  # 0 = full 180-frame test split; N = first N frames
SMOKE_LIMIT = 0  # 0 = all frames; N = N per split when converting

config = resolve_config()
PIPE_YAML, PIPE_PT = resolve_pipeline(PIPELINE_DIR)

print(f"Pipeline dir:  {PIPELINE_DIR}")
print(f"Pipeline YAML: {PIPE_YAML}")
print(f"Weights:       {PIPE_PT} ({PIPE_PT.stat().st_size / 1e6:.0f} MB)")
print(f"Test limit:    {TEST_LIMIT or 'full (180)'}")

## 1 · Load the trained pipeline

Register the Dinomaly plugin (so the saved node classes resolve), then
`CuvisPipeline.load_pipeline` rebuilds the graph and loads the weights.

In [ ]:
registry = NodeRegistry()
registry.register_plugin(str(config["plugins_yaml"]))
pipeline = CuvisPipeline.load_pipeline(
    str(PIPE_YAML), weights_path=str(PIPE_PT), device=str(device), node_registry=registry
)
pipeline.torch_layers.eval()
print("Loaded:", pipeline.name, "| nodes:", [n.name for n in pipeline.nodes])

SPLITS_JSON, UNIVERSE_CSV = prepare_lentils_data(
    Path("outputs/npz_local"),
    dataset_dir=Path("/content/data") if IN_COLAB else None,
    limit=SMOKE_LIMIT,
)
datamodule = MultiNpzDataModule(
    splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
    universe_csv=str(UNIVERSE_CSV),
    batch_size=1,
    num_workers=0,
)
datamodule.setup(stage="predict")  # Predictor evaluates the predict split (a copy of test)
print("Predict (= test) frames:", len(datamodule.predict_ds))

### Pipeline graph

Inline view of the restored graph. A `CuvisPipeline` renders itself in Jupyter as an
inline SVG, rendered from Graphviz DOT in memory (needs the system `dot` binary; it falls
back to a Mermaid source block otherwise).

In [ ]:
pipeline

## 2 · Run inference on the test split

`Predictor` runs the loaded pipeline over the split, so there is no hand-rolled forward loop. We
call it with `stage=ExecutionStage.TEST` so the pipeline's metric nodes fire (they are gated to
VAL/TEST); section 3 then reads their accumulated AUROC / IoU straight off the nodes. `collect_outputs=True`
hands back the per-frame model outputs (anomaly score map, per-frame score, selector RGB), which we
flatten into one record per frame for the per-class breakdown and qualitative panels below. The
DINOv2 encoder runs under `torch.no_grad()` inside `Predictor`. `class_mask` is not a pipeline
output, so it is reloaded from each frame's NPZ.

In [ ]:
def _port(batch_out, port) -> torch.Tensor | None:
    """Value of the first ``(node, port)`` key matching ``port`` in a Predictor batch output."""
    for (_node, name), value in batch_out.items():
        if name == port and value is not None:
            return value
    return None


def _frame(x, i) -> np.ndarray | None:
    """The i-th frame of a batched tensor as a float32 numpy array (None-safe)."""
    return None if x is None else x[i].detach().float().cpu().numpy()


collected = Predictor(pipeline, datamodule).predict(stage=ExecutionStage.TEST, collect_outputs=True)

# Flatten Predictor's per-batch (node, port) outputs into one record per frame. Model outputs come
# from the pipeline; the ground-truth class_mask is not a pipeline output, so reload it from the NPZ.
records = getattr(datamodule.predict_ds, "records", None) or getattr(
    datamodule.predict_ds, "_rows", None
)
results, offset = [], 0
for batch_out in collected:
    scores, ascore, rgb, mask = (
        _port(batch_out, p) for p in ("scores", "anomaly_score", "rgb_image", "mask")
    )
    bsz = int((scores if scores is not None else mask).shape[0])
    for i in range(bsz):
        rec = records[offset + i] if records is not None and offset + i < len(records) else {}
        path = rec.get("path") if isinstance(rec, dict) else None
        score = _frame(scores, i)
        if score is not None and score.ndim == 3 and score.shape[-1] == 1:
            score = score[..., 0]
        m = _frame(mask, i)
        results.append(
            {
                "score_map": None if score is None else score.astype(np.float32),
                "anomaly_score": None if ascore is None else float(ascore[i].item()),
                "mask": None if m is None else m.astype(np.int32),
                "class_mask": utils.load_lentils_frame(path)["class_mask"] if path else None,
                "rgb": _frame(rgb, i),
                "path": path,
                "index": rec.get("index") if isinstance(rec, dict) else None,
            }
        )
    offset += bsz
n_anom = sum(1 for r in results if r["mask"] is not None and r["mask"].any())
print(f"{len(results)} frames evaluated | {n_anom} anomalous, {len(results) - n_anom} normal")

## 3 · Overall metrics

Read straight off the pipeline's metric nodes, which accumulated during the `Predictor` pass above:
`metrics_auroc` (`AnomalyAUROCMetrics`) gives pixel + image AUROC, and `metrics_anomaly`
(`AnomalyDetectionMetrics`) gives IoU / F1 / average precision on the binarized map. The AUROC node
is histogram-binned at its `thresholds`, a close approximation to the exact pooled AUROC.

In [ ]:
by_name = {n.name: n for n in pipeline.nodes}
auroc, anomaly = by_name["metrics_auroc"], by_name["metrics_anomaly"]
print(f"pixel_auroc       : {float(auroc.pixel_auroc.compute()):.4f}")
print(f"image_auroc       : {float(auroc.image_auroc.compute()):.4f}")
print(f"iou               : {float(anomaly.iou_metric.compute()):.4f}")
print(f"f1                : {float(anomaly.f1_metric.compute()):.4f}")
print(f"average_precision : {float(anomaly.average_precision_metric.compute()):.4f}")

## 4 · Per-class pixel AUROC

For each non-background category, a one-vs-background pixel AUROC (this class's pixels as
positives, normal pixels as negatives), read straight from the baked `class_mask`. Classes
absent from the evaluated frames are skipped.

In [ ]:
scores = [r["score_map"] for r in results if r["score_map"] is not None]
cmasks = [r["class_mask"] for r in results if r["class_mask"] is not None]
per_class = utils.per_class_pixel_auroc(scores, cmasks, LENTILS_CATEGORIES)
for name, v in sorted(per_class.items(), key=lambda kv: kv[1]):
    print(f"{name:12s} {v:.4f}")
if per_class:
    utils.plot_per_class_auroc_bar(per_class, title="Lentils per-class pixel AUROC")
    plt.show()
else:
    print("(no non-background classes present in the evaluated frames)")

## 5 · Qualitative panels

A few anomalous frames: false-color scene, anomaly heatmap, and the scene with the
ground-truth contour overlaid. Cubes are reloaded from their NPZ for a true false-color view.

In [ ]:
anom = [r for r in results if r["mask"] is not None and r["mask"].any()][:3]
for r in anom:
    if r.get("path"):
        fr = utils.load_lentils_frame(r["path"])
        cube, wl = fr["cube"], fr["wavelengths"]
    else:  # fall back to the selector's RGB output
        cube = r["rgb"][None] if r["rgb"] is not None else None
        wl = np.array([650, 550, 450])
    title = f"index={r.get('index')}  score={r.get('anomaly_score')}"
    if cube is not None:
        utils.render_inference_panel(
            cube if cube.ndim == 3 else cube[0],
            r["score_map"],
            wavelengths=wl,
            gt_mask=r["mask"],
            title=title,
        )
        plt.show()

## Takeaways

- Numbers come from *your* trained pipeline: a 1-epoch smoke will look weak. For a real
  result, set `MAX_EPOCHS = 20` (or 50) in section 3 of `lentils_rgb_train_tutorial.ipynb`
  and rerun from section 6 (or launch the same run with `uv run restore-trainrun`).
- Per-class AUROC surfaces which foreign-object types Dinomaly separates best from normal
  lentils, useful for deciding where a supervised head would add the most.
- To compare front-ends, point `PIPELINE_DIR` (setup cell) at a CIR or AdaCLIP-bands run's
  `trained_models` dir and rerun on the identical test set.